# DFSL PPT Contrastive Cross Evaluation

This notebook is converted from `dfsl_ppt_contrastive.py` and keeps both requested additions:

- **Frequency augmentation** in the training dataset path.
- **Supervised contrastive loss** after the adaptive DFSL fusion embedding.

Workflow:

1. Train on **FaceForensics++** only.
2. Tune a small grid on the FF++ validation split.
3. Load the best FF++ checkpoint.
4. Cross-evaluate on **Celeb-DF** and **WildDeepfake**.
5. Save a final accuracy/AUC table.

In [ ]:
print('hi')

In [ ]:
import csv
import json
import math
import random
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

import timm
from datasets import load_dataset
from huggingface_hub import list_repo_files
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve

In [ ]:
# -------------------------
# Paths and configuration
# -------------------------
FFPP_ROOT = '/home/student/Dhara-Mtech/IPR/PROJECTS/PROJECTS/bs/bs-project-files/Dataset/FaceForensics++_C23'
CELEBDF_ROOT = '/home/student/Dhara-Mtech/IPR/PROJECTS/PROJECTS/bs/bs-project-files/Dataset/Celeb-DF v2'
WDF_REPO_ID = 'xingjunm/WildDeepfake'
OUT_DIR = Path('dfsl_ppt_cross_eval_out')
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
IMG_SIZE = 224
N_FRAMES = 4
BATCH_SIZE = 4
NUM_WORKERS = 4
BACKBONE = 'efficientnet_b4'
EMBED_DIM = 256
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 5
TEMPERATURE = 2.0
CONTRASTIVE_TEMPERATURE = 0.2
AMP = torch.cuda.is_available()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Training length. For real results use 30+ epochs; reduce for smoke tests.
EPOCHS = 30
RUN_TUNING = True

# Small tuning grid. It retrains each config on FF++ and picks best FF++ val AUC.
# For faster checking, keep only one config.
TUNE_GRID = [
    {'lr': 1.5e-4, 'dropout': 0.30, 'contrastive_weight': 0.10},
    {'lr': 1.0e-4, 'dropout': 0.30, 'contrastive_weight': 0.10},
    {'lr': 1.5e-4, 'dropout': 0.20, 'contrastive_weight': 0.05},
    {'lr': 7.5e-5, 'dropout': 0.20, 'contrastive_weight': 0.15},
]

# WDF cross-test size controls. Increase for stronger evaluation.
WDF_TARS_PER_CLASS_PER_SPLIT = 8
MAX_WDF_EXAMPLES_PER_SPLIT = None
MAX_WDF_SEQUENCES_PER_SPLIT = 2000
WDF_CLIP_STRIDE = 4
WDF_MAX_CLIPS_PER_SOURCE_SEQUENCE = 20

VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
print('device:', DEVICE)

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def find_videos(root):
    root = Path(root)
    return sorted(p for p in root.rglob('*') if p.suffix.lower() in VIDEO_EXTS)


def scan_ffpp(root):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f'FF++ root not found: {root}')
    rows = []
    for folder in sorted(p for p in root.iterdir() if p.is_dir()):
        name = folder.name.lower()
        label = 0 if ('real' in name or 'original' in name) else 1
        manip = 'real' if label == 0 else folder.name
        for video in find_videos(folder):
            rows.append({'kind': 'video', 'path': str(video), 'label': label, 'dataset': 'ffpp', 'manip': manip})
    if not rows:
        raise RuntimeError(f'No videos found under FF++ root: {root}')
    return rows


def scan_celebdf(root):
    root = Path(root)
    if not root.exists():
        print(f'Celeb-DF root not found: {root}. Celeb-DF cross-test will be skipped.')
        return []
    rows = []
    for folder in sorted(p for p in root.iterdir() if p.is_dir()):
        name = folder.name.lower()
        label = 0 if ('real' in name or 'youtube' in name) else 1
        for video in find_videos(folder):
            rows.append({'kind': 'video', 'path': str(video), 'label': label, 'dataset': 'celebdf', 'manip': folder.name})
    print(f'Celeb-DF videos={len(rows)} real={sum(r["label"] == 0 for r in rows)} fake={sum(r["label"] == 1 for r in rows)}')
    return rows


def stratified_split(rows, val_ratio=0.1, test_ratio=0.1, seed=42):
    rng = random.Random(seed)
    by_label = defaultdict(list)
    for row in rows:
        by_label[row['label']].append(dict(row))
    train, val, test = [], [], []
    for label_rows in by_label.values():
        rng.shuffle(label_rows)
        n = len(label_rows)
        n_val = max(1, int(n * val_ratio)) if val_ratio > 0 else 0
        n_test = max(1, int(n * test_ratio)) if test_ratio > 0 else 0
        val.extend(label_rows[:n_val])
        test.extend(label_rows[n_val:n_val + n_test])
        train.extend(label_rows[n_val + n_test:])
    rng.shuffle(train)
    rng.shuffle(val)
    rng.shuffle(test)
    return train, val, test

In [ ]:
def _wdf_label_from_example(example):
    key = str(example.get('__key__', '')).lower().replace('\\', '/')
    url = str(example.get('__url__', '')).lower().replace('\\', '/')
    text = f'/{key}/ /{url}/'
    if any(m in text for m in ['/real/', '/real_train/', '/real_test/', '_real/', '-real/']):
        return 0
    if any(m in text for m in ['/fake/', '/fake_train/', '/fake_test/', '_fake/', '-fake/']):
        return 1
    raise ValueError(f"Could not infer WDF label from __key__={example.get('__key__')} __url__={example.get('__url__')}")


def _wdf_sequence_id(example):
    key = str(example.get('__key__', ''))
    parts = [p for p in key.replace('\\', '/').split('/') if p and p != '.']
    if len(parts) >= 2:
        return '/'.join(parts[:-1])
    return key.rsplit('.', 1)[0]


def _natural_tar_key(path):
    stem = Path(path).name.replace('.tar.gz', '')
    return int(stem) if stem.isdigit() else stem


def _build_explicit_wdf_data_files(repo_id=WDF_REPO_ID, tars_per_class_per_split=WDF_TARS_PER_CLASS_PER_SPLIT):
    files = list_repo_files(repo_id, repo_type='dataset')
    folders = {
        'train': ['deepfake_in_the_wild/real_train/', 'deepfake_in_the_wild/fake_train/'],
        'test': ['deepfake_in_the_wild/real_test/', 'deepfake_in_the_wild/fake_test/'],
    }
    data_files = {'train': [], 'test': []}
    for split, prefixes in folders.items():
        for prefix in prefixes:
            shard_files = sorted([f for f in files if f.startswith(prefix) and f.endswith('.tar.gz')], key=_natural_tar_key)
            if tars_per_class_per_split is not None:
                shard_files = shard_files[:tars_per_class_per_split]
            urls = [f'hf://datasets/{repo_id}/{f}' for f in shard_files]
            data_files[split].extend(urls)
            print(f'WDF explicit {split}-{"real" if "real_" in prefix else "fake"} tar files:', len(urls))
    return data_files


def load_wdf_dataset(repo_id=WDF_REPO_ID):
    data_files = _build_explicit_wdf_data_files(repo_id)
    ds = load_dataset('webdataset', data_files=data_files)
    print(ds)
    return ds


def load_wdf_clips(hf_dataset, n_frames=4, split_name='test'):
    rows = []
    image_cols = {'png', 'jpg', 'jpeg', 'image'}
    ds = hf_dataset[split_name]
    meta_cols_to_remove = [c for c in ds.column_names if c in image_cols]
    meta_ds = ds.remove_columns(meta_cols_to_remove) if meta_cols_to_remove else ds
    limit = len(meta_ds) if MAX_WDF_EXAMPLES_PER_SPLIT is None else min(len(meta_ds), MAX_WDF_EXAMPLES_PER_SPLIT)
    grouped = defaultdict(list)
    for idx in tqdm(range(limit), desc=f'index WDF {split_name}'):
        ex = meta_ds[idx]
        label = _wdf_label_from_example(ex)
        seq = _wdf_sequence_id(ex)
        grouped[(label, seq)].append(idx)

    quota_per_label = None if MAX_WDF_SEQUENCES_PER_SPLIT is None else max(1, MAX_WDF_SEQUENCES_PER_SPLIT // 2)
    split_rows = []
    for (label, seq), indices in grouped.items():
        indices = sorted(indices)
        if len(indices) < n_frames:
            continue
        starts = list(range(0, len(indices) - n_frames + 1, max(1, WDF_CLIP_STRIDE))) or [0]
        if WDF_MAX_CLIPS_PER_SOURCE_SEQUENCE is not None and len(starts) > WDF_MAX_CLIPS_PER_SOURCE_SEQUENCE:
            starts = np.linspace(0, len(indices) - n_frames, WDF_MAX_CLIPS_PER_SOURCE_SEQUENCE).astype(int).tolist()
        for clip_id, start in enumerate(starts):
            split_rows.append({
                'kind': 'hf_frames', 'hf_split': split_name, 'indices': indices[start:start+n_frames],
                'label': label, 'dataset': 'wdf', 'sequence': f'{seq}#clip{clip_id}'
            })
    rng = random.Random(SEED)
    for label in [0, 1]:
        xs = [r for r in split_rows if r['label'] == label]
        rng.shuffle(xs)
        if quota_per_label is not None:
            xs = xs[:quota_per_label]
        rows.extend(xs)
    rng.shuffle(rows)
    print(f'WDF {split_name} clips={len(rows)} real={sum(r["label"] == 0 for r in rows)} fake={sum(r["label"] == 1 for r in rows)}')
    return rows

In [ ]:
class FrequencyAugment:
    """Random FFT high-frequency gain perturbation. Training only."""
    def __init__(self, p=0.35, gain_range=(0.75, 1.35)):
        self.p = p
        self.gain_range = gain_range

    def __call__(self, img):
        if random.random() > self.p:
            return img
        x = img.astype(np.float32) / 255.0
        h, w = x.shape[:2]
        yy, xx = np.ogrid[:h, :w]
        cy, cx = h // 2, w // 2
        radius = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
        high = radius > random.uniform(0.12, 0.28) * min(h, w)
        gain = random.uniform(*self.gain_range)
        out = np.empty_like(x)
        for c in range(3):
            spec = np.fft.fftshift(np.fft.fft2(x[..., c]))
            spec[high] *= gain
            out[..., c] = np.fft.ifft2(np.fft.ifftshift(spec)).real
        return np.clip(out * 255.0, 0, 255).astype(np.uint8)


class FaceCropper:
    def __init__(self, size, margin=0.25):
        self.size = size
        cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
        self.detector = cv2.CascadeClassifier(cascade_path)
        self.margin = margin

    def __call__(self, rgb):
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        faces = self.detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4, minSize=(48, 48))
        if len(faces) == 0:
            return cv2.resize(rgb, (self.size, self.size))
        x, y, w, h = max(faces, key=lambda b: b[2] * b[3])
        pad = int(max(w, h) * self.margin)
        x1, y1 = max(0, x - pad), max(0, y - pad)
        x2, y2 = min(rgb.shape[1], x + w + pad), min(rgb.shape[0], y + h + pad)
        return cv2.resize(rgb[y1:y2, x1:x2], (self.size, self.size))


def read_video_clip(path, n_frames, train, img_size, cropper):
    cap = cv2.VideoCapture(str(path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None
    if train:
        start = random.randint(0, max(0, total - n_frames))
        indices = list(range(start, min(total, start + n_frames)))
    else:
        indices = np.linspace(0, max(0, total - 1), n_frames).astype(int).tolist()
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, bgr = cap.read()
        if not ok:
            continue
        frames.append(cropper(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)))
    cap.release()
    if not frames:
        return None
    while len(frames) < n_frames:
        frames.append(frames[-1])
    return frames[:n_frames]


def _image_from_hf_example(example):
    for key in ('png', 'jpg', 'jpeg', 'image'):
        if key in example and example[key] is not None:
            img = example[key]
            if isinstance(img, Image.Image):
                return img.convert('RGB')
            return Image.open(img).convert('RGB')
    raise KeyError(f'No image column found. Columns: {list(example.keys())}')


def read_hf_frame_sequence(hf_dataset, split, indices, img_size):
    frames = []
    ds = hf_dataset[split]
    for idx in indices:
        img = _image_from_hf_example(ds[int(idx)]).resize((img_size, img_size))
        frames.append(np.asarray(img))
    return frames


class ForgeryClipDataset(Dataset):
    def __init__(self, rows, img_size=224, n_frames=4, train=False, hf_dataset=None):
        self.rows = rows
        self.img_size = img_size
        self.n_frames = n_frames
        self.train = train
        self.hf_dataset = hf_dataset
        self.cropper = FaceCropper(img_size)
        self.freq_aug = FrequencyAugment(p=0.35 if train else 0.0)
        self.tf = T.Compose([
            T.ToPILImage(),
            T.RandomHorizontalFlip() if train else nn.Identity(),
            T.ColorJitter(0.15, 0.15, 0.08, 0.02) if train else nn.Identity(),
            T.ToTensor(),
            T.Normalize(MEAN, STD),
        ])

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        if row.get('kind') == 'hf_frames':
            frames = read_hf_frame_sequence(self.hf_dataset, row['hf_split'], row['indices'], self.img_size)
        else:
            frames = read_video_clip(row['path'], self.n_frames, self.train, self.img_size, self.cropper)
        if frames is None:
            frames = [np.zeros((self.img_size, self.img_size, 3), np.uint8) for _ in range(self.n_frames)]
        frames = [self.freq_aug(f) for f in frames]
        clip = torch.stack([self.tf(f) for f in frames])
        return clip, int(row['label']), row.get('dataset', 'unknown')

In [ ]:
def thumbnail_layout(clip):
    f0, f1, f2, f3 = clip[:, 0], clip[:, 1], clip[:, 2], clip[:, 3]
    return torch.cat([torch.cat([f0, f1], dim=-1), torch.cat([f3, f2], dim=-1)], dim=-2)


class MS3(nn.Module):
    def __init__(self, ch, drop):
        super().__init__()
        self.norm = nn.LayerNorm(ch)
        self.mix = nn.Sequential(
            nn.Conv1d(ch, ch, 7, padding=3, groups=ch, bias=False),
            nn.Conv1d(ch, ch * 2, 1, bias=False),
            nn.GLU(dim=1),
            nn.Dropout(drop),
        )
        self.fuse = nn.Sequential(nn.Conv2d(ch * 4, ch, 1, bias=False), nn.BatchNorm2d(ch), nn.SiLU())

    def scan(self, x):
        b, c, h, w = x.shape
        return self.mix(x.flatten(2)).view(b, c, h, w)

    def forward(self, x):
        z = self.norm(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)
        d0 = self.scan(z)
        d1 = torch.flip(self.scan(torch.flip(z, [-1])), [-1])
        d2 = self.scan(z.transpose(-1, -2)).transpose(-1, -2)
        d3 = torch.flip(self.scan(torch.flip(z.transpose(-1, -2), [-1])), [-1]).transpose(-1, -2)
        return x + self.fuse(torch.cat([d0, d1, d2, d3], dim=1))


class FourierFGA(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.proj = nn.Conv2d(1, ch, 1)

    def forward(self, feat):
        spec = torch.fft.fftshift(torch.fft.fft2(feat.float(), norm='ortho'), dim=(-2, -1))
        mag = torch.log1p(spec.abs()).mean(1, keepdim=True)
        mag = (mag - mag.amin((-2, -1), keepdim=True)) / (mag.amax((-2, -1), keepdim=True) - mag.amin((-2, -1), keepdim=True) + 1e-6)
        return feat * torch.sigmoid(self.proj(mag.to(feat.dtype)))


class WaveletFGA(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.proj = nn.Conv2d(1, ch, 1)

    def forward(self, feat):
        a, b = feat[..., 0::2, 0::2], feat[..., 0::2, 1::2]
        c, d = feat[..., 1::2, 0::2], feat[..., 1::2, 1::2]
        high = ((a - b + c - d).abs() + (a + b - c - d).abs() + (a - b - c + d).abs()).mean(1, keepdim=True)
        high = F.interpolate(high, feat.shape[-2:], mode='bilinear', align_corners=False)
        high = (high - high.amin((-2, -1), keepdim=True)) / (high.amax((-2, -1), keepdim=True) - high.amin((-2, -1), keepdim=True) + 1e-6)
        return feat * torch.sigmoid(self.proj(high))


class STL(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.fuse = nn.Sequential(nn.Conv2d(ch, ch, 1, bias=False), nn.BatchNorm2d(ch), nn.SiLU())

    def forward(self, frame_feats):
        out = []
        for masked in range(len(frame_feats)):
            kept = [f for i, f in enumerate(frame_feats) if i != masked]
            out.append(self.fuse(torch.stack(kept).mean(0)))
        return out


class DFSLNet(nn.Module):
    def __init__(self, backbone='efficientnet_b4', img_size=224, embed_dim=256, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=True, features_only=True, out_indices=[-1])
        with torch.no_grad():
            ch_in = self.backbone(torch.zeros(1, 3, img_size, img_size))[0].shape[1]
        self.neck = nn.Sequential(
            nn.AdaptiveAvgPool2d(16),
            nn.Conv2d(ch_in, embed_dim, 1, bias=False),
            nn.BatchNorm2d(embed_dim),
            nn.SiLU(),
        )
        self.ms3_global = nn.Sequential(MS3(embed_dim, dropout), MS3(embed_dim, dropout))
        self.ms3_local = MS3(embed_dim, dropout)
        self.fourier = FourierFGA(embed_dim)
        self.wavelet = WaveletFGA(embed_dim)
        self.stl = STL(embed_dim)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fusion_logits = nn.Parameter(torch.zeros(2))
        self.fusion_proj = nn.Sequential(nn.Linear(embed_dim, embed_dim), nn.SiLU(), nn.Dropout(dropout * 0.5))
        self.head_g = self._head(embed_dim, embed_dim // 2, dropout)
        self.head_gz = self._head(embed_dim, embed_dim // 4, dropout)
        self.head_l = self._head(embed_dim, embed_dim // 2, dropout)
        self.head = self._head(embed_dim, embed_dim // 2, dropout)

    @staticmethod
    def _head(in_ch, mid_ch, dropout):
        return nn.Sequential(nn.Dropout(dropout), nn.Linear(in_ch, mid_ch), nn.SiLU(), nn.Dropout(dropout * 0.5), nn.Linear(mid_ch, 2))

    def extract(self, img):
        return self.neck(self.backbone(img)[0])

    def forward(self, clip):
        b, t, c, h, w = clip.shape
        layout = F.interpolate(thumbnail_layout(clip), (h, w), mode='bilinear', align_corners=False)
        g = self.fourier(self.ms3_global(self.extract(layout)))
        gvec = self.pool(g).flatten(1)

        frame_feats = [self.ms3_local(self.extract(clip[:, i])) for i in range(t)]
        local_vecs, local_logits = [], []
        for lf in self.stl(frame_feats):
            lf = self.wavelet(lf)
            lv = self.pool(lf).flatten(1)
            local_vecs.append(lv)
            local_logits.append(self.head_l(lv))
        lvec = torch.stack(local_vecs).mean(0)
        fusion_w = torch.softmax(self.fusion_logits, dim=0)
        fused = self.fusion_proj(fusion_w[0] * gvec + fusion_w[1] * lvec)
        return {
            'logits': self.head(fused),
            'logits_g': self.head_g(gvec),
            'logits_gz': self.head_gz(gvec),
            'local_list': local_logits,
            'embedding': fused,
            'fusion_weights': fusion_w.detach(),
        }

In [ ]:
class DFSLContrastiveLoss(nn.Module):
    def __init__(self, temperature=2.0, contrastive_weight=0.1, contrastive_temperature=0.2):
        super().__init__()
        self.temperature = temperature
        self.contrastive_weight = contrastive_weight
        self.contrastive_temperature = contrastive_temperature
        self.log_var = nn.Parameter(torch.zeros(3))

    def weight(self, i):
        s2 = self.log_var[i].exp()
        return 0.5 / s2 + torch.log1p(s2)

    def soft_kl(self, student, teacher):
        t = self.temperature
        return F.kl_div(F.log_softmax(student / t, dim=1), F.softmax(teacher.detach() / t, dim=1), reduction='batchmean') * (t ** 2)

    def supervised_contrastive(self, emb, labels):
        emb = F.normalize(emb.float(), dim=1)
        sim = emb @ emb.t() / self.contrastive_temperature
        sim = sim - sim.max(dim=1, keepdim=True).values.detach()
        eye = torch.eye(labels.numel(), dtype=torch.bool, device=labels.device)
        same = labels[:, None].eq(labels[None, :]) & ~eye
        if not same.any():
            return emb.new_zeros(())
        log_prob = sim - torch.logsumexp(sim.masked_fill(eye, -1e9), dim=1, keepdim=True)
        loss = -(log_prob * same.float()).sum(1) / same.float().sum(1).clamp_min(1.0)
        return loss[same.any(1)].mean()

    def forward(self, out, labels):
        ce = F.cross_entropy(out['logits'], labels)
        gil = self.soft_kl(out['logits_gz'], out['logits_g'])
        locs = out['local_list']
        lil = sum(self.soft_kl(locs[i], locs[j]) for i in range(len(locs)) for j in range(len(locs)) if i != j)
        lil = lil / max(1, len(locs) * (len(locs) - 1))
        contrast = self.supervised_contrastive(out['embedding'], labels)
        return self.weight(0) * ce + self.weight(1) * gil + self.weight(2) * lil + self.contrastive_weight * contrast


def make_loader(rows, train=False, hf_dataset=None, batch_size=None):
    ds = ForgeryClipDataset(rows, IMG_SIZE, N_FRAMES, train, hf_dataset=hf_dataset)
    batch_size = batch_size or BATCH_SIZE
    if train:
        labels = [r['label'] for r in rows]
        counts = {c: max(1, labels.count(c)) for c in [0, 1]}
        weights = [len(labels) / (2.0 * counts[l]) for l in labels]
        sampler = WeightedRandomSampler(weights, num_samples=len(labels), replacement=True)
        return DataLoader(ds, batch_size=batch_size, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)


def cosine_lr(epoch, total_epochs, warmup_epochs, base_lr):
    if epoch < warmup_epochs:
        return base_lr * float(epoch + 1) / max(1, warmup_epochs)
    p = float(epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * p))


def compute_metrics(labels, scores):
    labels = np.asarray(labels, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    bad = ~np.isfinite(scores)
    if bad.any():
        scores = np.nan_to_num(scores, nan=0.5, posinf=1.0, neginf=0.0)
    scores = np.clip(scores, 0.0, 1.0)
    preds = (scores >= 0.5).astype(np.int64)
    acc = accuracy_score(labels, preds) * 100.0
    if len(set(labels.tolist())) == 2:
        auc = roc_auc_score(labels, scores) * 100.0
        fpr, tpr, _ = roc_curve(labels, scores)
        eer = fpr[np.nanargmin(np.abs((1 - tpr) - fpr))] * 100.0
    else:
        auc, eer = float('nan'), float('nan')
    return {'acc': float(acc), 'auc': float(auc), 'eer': float(eer), 'bad_scores': int(bad.sum())}


@torch.no_grad()
def evaluate(model, loader, device=DEVICE):
    model.eval()
    labels, scores = [], []
    for clips, y, _ in tqdm(loader, desc='eval', leave=False):
        clips = clips.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=AMP):
            out = model(clips)
        logits = torch.nan_to_num(out['logits'].float(), nan=0.0, posinf=50.0, neginf=-50.0).clamp(-50, 50)
        scores.extend(F.softmax(logits, dim=1)[:, 1].detach().cpu().tolist())
        labels.extend(y.tolist())
    return compute_metrics(labels, scores)

In [ ]:
def train_ffpp_config(cfg, ffpp_train, ffpp_val, run_dir):
    seed_everything(SEED)
    run_dir.mkdir(parents=True, exist_ok=True)
    train_loader = make_loader(ffpp_train, train=True, batch_size=cfg['batch_size'])
    val_loader = make_loader(ffpp_val, train=False, batch_size=cfg['batch_size'])

    model = DFSLNet(cfg['backbone'], IMG_SIZE, EMBED_DIM, cfg['dropout']).to(DEVICE)
    criterion = DFSLContrastiveLoss(TEMPERATURE, cfg['contrastive_weight'], CONTRASTIVE_TEMPERATURE).to(DEVICE)
    optimizer = torch.optim.Adam(
        [{'params': model.parameters(), 'lr': cfg['lr']}, {'params': criterion.parameters(), 'lr': cfg['lr']}],
        weight_decay=WEIGHT_DECAY,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=AMP)
    best_auc = -1.0
    history = []
    ckpt_path = run_dir / 'best.pt'

    for epoch in range(cfg['epochs']):
        lr = cosine_lr(epoch, cfg['epochs'], WARMUP_EPOCHS, cfg['lr'])
        for group in optimizer.param_groups:
            group['lr'] = lr
        model.train()
        losses = []
        for clips, y, _ in tqdm(train_loader, desc=f"{run_dir.name} epoch {epoch+1}/{cfg['epochs']}", leave=False):
            clips = clips.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=AMP):
                out = model(clips)
                loss = criterion(out, y)
            if not torch.isfinite(loss):
                print('Skipping non-finite loss:', float(loss.detach().cpu()))
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(list(model.parameters()) + list(criterion.parameters()), 1.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))

        val_m = evaluate(model, val_loader)
        row = {'epoch': epoch + 1, 'loss': float(np.mean(losses)) if losses else float('nan'), 'lr': lr, **{f'val_{k}': v for k, v in val_m.items()}}
        history.append(row)
        print(row)
        score = val_m['auc'] if not np.isnan(val_m['auc']) else val_m['acc']
        if score > best_auc:
            best_auc = score
            torch.save({'model': model.state_dict(), 'criterion': criterion.state_dict(), 'cfg': cfg, 'val': val_m}, ckpt_path)

    pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
    return {'cfg': cfg, 'best_val_auc': best_auc, 'ckpt': str(ckpt_path), 'history': history}


def load_model_from_ckpt(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    cfg = ckpt['cfg']
    model = DFSLNet(cfg['backbone'], IMG_SIZE, EMBED_DIM, cfg['dropout']).to(DEVICE)
    model.load_state_dict(ckpt['model'])
    return model, cfg

In [ ]:
# Load FF++, Celeb-DF, and WDF cross-test rows.
seed_everything(SEED)
ffpp_rows = scan_ffpp(FFPP_ROOT)
ffpp_train, ffpp_val, ffpp_test = stratified_split(ffpp_rows, val_ratio=0.1, test_ratio=0.1, seed=SEED)
celeb_rows = scan_celebdf(CELEBDF_ROOT)

wdf_dataset = load_wdf_dataset(WDF_REPO_ID)
wdf_test = load_wdf_clips(wdf_dataset, n_frames=N_FRAMES, split_name='test')

print('FF++ train/val/test:', len(ffpp_train), len(ffpp_val), len(ffpp_test))
print('FF++ train real/fake:', sum(r['label']==0 for r in ffpp_train), sum(r['label']==1 for r in ffpp_train))
print('Celeb-DF test:', len(celeb_rows), 'real=', sum(r['label']==0 for r in celeb_rows), 'fake=', sum(r['label']==1 for r in celeb_rows))
print('WDF test:', len(wdf_test), 'real=', sum(r['label']==0 for r in wdf_test), 'fake=', sum(r['label']==1 for r in wdf_test))

In [ ]:
# Train on FF++ and tune using FF++ validation AUC.
base_cfg = {
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'backbone': BACKBONE,
}
configs = []
if RUN_TUNING:
    for item in TUNE_GRID:
        cfg = dict(base_cfg)
        cfg.update(item)
        configs.append(cfg)
else:
    cfg = dict(base_cfg)
    cfg.update(TUNE_GRID[0])
    configs = [cfg]

train_results = []
for i, cfg in enumerate(configs):
    run_dir = OUT_DIR / f"ffpp_tune_{i}_lr{cfg['lr']}_drop{cfg['dropout']}_cw{cfg['contrastive_weight']}"
    print('CONFIG', i, json.dumps(cfg, indent=2))
    result = train_ffpp_config(cfg, ffpp_train, ffpp_val, run_dir)
    train_results.append(result)
    with (OUT_DIR / 'tuning_partial.json').open('w') as f:
        json.dump(train_results, f, indent=2)

train_results = sorted(train_results, key=lambda r: r['best_val_auc'], reverse=True)
best = train_results[0]
print('BEST:', json.dumps(best['cfg'], indent=2), 'val_auc=', best['best_val_auc'])

In [ ]:
# Final cross evaluation: model trained only on FF++, tested on FF++, Celeb-DF, and WDF.
best_model, best_cfg = load_model_from_ckpt(best['ckpt'])
loaders = {
    'FF++ holdout': make_loader(ffpp_test, train=False, batch_size=best_cfg['batch_size']),
}
if celeb_rows:
    loaders['Celeb-DF cross'] = make_loader(celeb_rows, train=False, batch_size=best_cfg['batch_size'])
if wdf_test:
    loaders['WDF cross'] = make_loader(wdf_test, train=False, hf_dataset=wdf_dataset, batch_size=best_cfg['batch_size'])

summary = []
for name, loader in loaders.items():
    m = evaluate(best_model, loader)
    row = {'train_dataset': 'FF++', 'test_dataset': name, **m, 'checkpoint': best['ckpt']}
    summary.append(row)
    print(name, m)

summary_df = pd.DataFrame(summary)
summary_df.to_csv(OUT_DIR / 'cross_eval_summary.csv', index=False)
(OUT_DIR / 'cross_eval_summary.json').write_text(summary_df.to_json(orient='records', indent=2))
summary_df